# 04 - Analyse de la Masse des Neutrinos: LambdaCDM vs Tachyon

Ce notebook a pour objectif de vérifier si le champ Tachyon permet de résoudre l'anomalie cosmologique récente sur la masse des neutrinos. Sous le modèle standard LambdaCDM, les données récentes ont tendance à "écraser" la masse des neutrinos vers des valeurs impossibles en physique des particules (proches de 0 eV, voire négatives).

Nous allons vérifier si l'introduction du modèle géométrique de Souriau (le champ Tachyon) absorbe cette tension et permet aux neutrinos de retrouver une masse physiquement valide (supérieure à la limite expérimentale de ~0.06 eV).

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, MCSamples, loadMCSamples

plt.rcParams.update({'figure.dpi': 120, 'font.size': 14,'axes.grid': True, 'grid.alpha': 0.3})

print("Entorno preparado para análisis de Neutrinos.")

Entorno preparado para análisis de Neutrinos.


## 1. Préparation de l'environnement de travail

Dans cette première étape, nous préparons notre environnement Python. Nous utilisons GetDist, la librairie de référence en cosmologie développée pour analyser et tracer les résultats des chaînes de Markov (MCMC). Elle gère nativement la pondération (les weights) et le lissage des distributions statistiques.

In [10]:
# CAMBIAR
ruta_tachyon = '../../cobaya_runs/tachyon_output/run05_expo_pantheon_full/full_expo_pantheon'
ruta_lcdm = '/content/drive/MyDrive/souriau_chains/full_expo_pantheon/full_expo_pantheon'

print("Cargando muestras LambdaCDM...")
muestras_lcdm = loadMCSamples(ruta_lcdm, settings={'ignore_rows': 0.3})

print("Cargando muestras Tachyon...")
muestras_tach = loadMCSamples(ruta_tachyon, settings={'ignore_rows': 0.3})

PARAM_NEUTRINO = 'mnu' # como se guarda la masa dle neutrino en el .yaml

Cargando muestras LambdaCDM...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/souriau_chains/full_expo_pantheon'

## 2. Chargement des Chaînes MCMC et Application du Burn-in

Nous importons ici les données brutes générées par nos exécutions sur Cobaya (le run de référence LambdaCDM et notre run Tachyon).

Pourquoi appliquer un "burn-in" de 30%?
Lorsqu'un algorithme MCMC démarre, il passe ses premiers milliers de pas à "chercher" la vallée où les paramètres sont optimaux. Cette phase d'exploration initiale ne représente pas la vraie probabilité finale. En Data Science bayésienne, il est impératif de couper (ignorer) ce début de chaîne pour ne calculer nos statistiques que sur la marche aléatoire stable (es un mero primer approach al espacio de búsqueda).

In [9]:
stats_lcdm = muestras_lcdm.getMargeStats()
stats_tach = muestras_tach.getMargeStats()

media_lcdm = stats_lcdm.parWithName(PARAM_NEUTRINO).mean
err_lcdm = stats_lcdm.parWithName(PARAM_NEUTRINO).err

media_tach = stats_tach.parWithName(PARAM_NEUTRINO).mean
err_tach = stats_tach.parWithName(PARAM_NEUTRINO).err

print("RESULTADOS DEL SHIFT DE NEUTRINOS")
print(f"Modelo ΛCDM      : Σm_ν = {media_lcdm:.4f} ± {err_lcdm:.4f} eV")
print(f"Modelo Taquión   : Σm_ν = {media_tach:.4f} ± {err_tach:.4f} eV")

shift = media_tach - media_lcdm
print(f"SHIFT DETECTADO: {shift:+.4f} eV")

# Lógica de decisión teórica de Souriau
if media_lcdm < 0.03 and media_tach > 0.05:
    print("\n¡SANTO GRIAL ALCANZADO!")
    print("Bajo ΛCDM, los datos forzaban masas no-físicas (cerca de 0).")
    print("El campo del Taquión ha absorbido la anomalía, permitiendo que")
    print("los neutrinos recuperen una masa física real (> 0.06 eV).")
    print("Esto es evidencia empírica directa para la teoría geométrica.")


NameError: name 'muestras_lcdm' is not defined

## 3. Preuve Empirique de la Théorie

Dans la prochaine cellule nous testons numériquement l'hypothèse de la "couche basse" de la théorie de Souriau.
Nous extrayons la médiane et l'erreur (les statistiques marginalisées) pour la somme des masses des neutrinos dans les deux univers simulés.

Notre algorithme inclut une règle de décision stricte: si la masse médiane sous LambdaCDM est poussée sous les 0.03 eV (irréaliste), mais que le passage au modèle Tachyon permet à la masse de "respirer" et de remonter au-dessus de 0.05 / 0.06 eV (limites empiriques), nous obtenons la preuve statistique que le champ Tachyon absorbe la pathologie du modèle standard.

In [ ]:
g = plots.get_subplot_plotter()

params_comparacion = ['H0', 'Omega_m', PARAM_NEUTRINO]

g.triangle_plot(
    [muestras_lcdm, muestras_tach],
    params_comparacion,
    filled=True,
    legend_labels=['Modèle $\\Lambda$CDM', 'Modèle Tachyon (Souriau)'],
    contour_colors=['#1f77b4', '#d62728'],
    title_limit=1 # Esto pone los valores medios encima de la curva 1D
)

# Dibujar la línea de 0.06 eV (Límite físico de experimentos de oscilación)
for i in range(len(params_comparacion)):
    ax = g.subplots[i, i]
    if ax is not None and params_comparacion[i] == PARAM_NEUTRINO:
        ax.axvline(0.06, color='gray', linestyle='--', lw=2, label='Límite Exp (0.06 eV)')
        ax.legend(loc='upper right', fontsize=10)

plt.suptitle("Effet du Champ Tachyonique sur la Masse des Neutrinos", fontsize=18, y=1.05)
plt.savefig('/results/figures/corner_neutrinos_shift.png', bbox_inches='tight', dpi=300)
plt.show()

## 4. Visualisation Globale : Le "Corner Plot" Comparatif

Ce type de graphique superpose les distributions de probabilité a posteriori (les "cloches") en 1D, et les ellipses de corrélation en 2D.

Nous traçons une ligne de démarcation physique stricte à 0.06 eV, qui correspond à la limite basse mesurée par les expériences d'oscillation de particules sur Terre. L'objectif visuel est d'observer la cloche bleue (LambdaCDM) s'écraser à gauche de cette ligne, tandis que la cloche rouge (Tachyon) bascule naturellement à droite, dans la zone physiquement autorisée.

In [ ]:
param_taquion = 'tachyon_lambda'

if param_taquion in muestras_tach.getParamNames().list():
    g2 = plots.get_single_plotter(width_inch=6)
    g2.plot_2d(muestras_tach, param_taquion, PARAM_NEUTRINO, filled=True, colors=['#d62728'])
    
    g2.add_y_marker(0.06, color='gray', ls='--', lw=2)
    
    plt.title("Dégénérescence : Tachyon $\\lambda$ vs $\\Sigma m_\\nu$")
    plt.savefig('/content/drive/MyDrive/souriau_chains/degeneracy_tachyon_nu.png', bbox_inches='tight', dpi=300)
    plt.show()